# Module 4 — Text mining : applications

**CONSEIL :** Sauvegarde une copie de ce notebook dans ton Drive avant de commencer !

**Objectifs d'apprentissage**
- Enchaîner **cinq applications** concrètes du text mining, de la plus simple à la plus avancée
- Décrire un corpus avec un **nuage de mots** et des **mots-clés** distinctifs
- Mesurer un **sentiment** avec un modèle pré-entraîné
- Entraîner une **classification supervisée** (ML) et lire ses résultats
- Faire de la **recherche sémantique** par embeddings
- Repérer les **biais éthiques** des modèles et des corpus

## Le corpus : critiques de films notées (Allociné)

On travaille sur un échantillon de **critiques de films** rédigées sur Allociné, chacune étiquetée **positif** ou **négatif** d'après la note laissée par l'internaute. Un corpus idéal pour le text mining : du texte libre, en français, avec une **étiquette** qui permet d'illustrer aussi bien les méthodes descriptives que l'apprentissage supervisé.

| Fichier | Lignes | Équilibre |
|---|---|---|
| `critiques_films_train.csv` | 3 000 | 1 500 positifs / 1 500 négatifs |
| `critiques_films_test.csv` | 1 000 | 500 positifs / 500 négatifs |

## Setup

In [ ]:
# Sur Colab, décommente la ligne suivante pour installer les dépendances :
# !pip install -q scikit-learn transformers sentence-transformers wordcloud matplotlib pandas

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Les données se chargent **toutes seules**, sans aucun fichier à uploader : on lit la copie hébergée si elle est disponible, sinon on reconstruit l'échantillon **directement depuis la source** ([Allociné sur Hugging Face](https://huggingface.co/datasets/tblard/allocine)).

In [ ]:
import io, requests

def telecharger_allocine(split, n_par_classe):
    url = (f"https://huggingface.co/datasets/tblard/allocine/resolve/main/"
           f"allocine/{split}-00000-of-00001.parquet")
    brut = pd.read_parquet(io.BytesIO(requests.get(url, timeout=180).content))
    parts = [g.sample(min(n_par_classe, len(g)), random_state=42)
             for _, g in brut.groupby("label")]
    out = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)
    out = out.rename(columns={"review": "text", "label": "polarite"})
    out["polarite"] = out["polarite"].map({0: "négatif", 1: "positif"})
    return out[["text", "polarite"]]

def charger(split, n_par_classe):
    local = f"../../../data/critiques_films/critiques_films_{split}.csv"
    url = ("https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/"
           f"data/critiques_films/critiques_films_{split}.csv")
    try:
        return pd.read_csv(local if os.path.exists(local) else url)
    except Exception:
        print(f"Reconstruction du split '{split}' depuis Hugging Face...")
        return telecharger_allocine(split, n_par_classe)

train = charger("train", 1500)
test = charger("test", 500)

print(f"Train : {len(train):,} | Test : {len(test):,}")
train.head(3)

On aura besoin d'une liste de **stopwords français** (mots outils à ignorer). `scikit-learn` n'en fournit pas, on en définit une courte ici.

In [ ]:
STOP_FR = set('''
au aux avec ce ces dans de des du elle en et eux il je la le les leur lui ma
mais me même mes moi mon ne nos notre nous on ou par pas pour qu que qui sa se
ses son sur ta te tes toi ton tu un une vos votre vous c d j l à m n s t y été
être ai as avait avais avons ont est sont a als cette cet plus moins très bien
fait film films très tout tous toute toutes plus comme aussi car donc alors
sans sous entre vers chez ni leurs cela ça là où dont quand
'''.split())
print(len(STOP_FR), "stopwords")

## 4.1 Application 1 — Nuage de mots et mots-clés distinctifs

*La plus simple : descriptive, sans aucun modèle.* On compte les mots et on regarde lesquels distinguent le mieux les bonnes des mauvaises critiques. Pour ça, le **log-odds** compare la fréquence d'un mot chez les positifs et chez les négatifs : très positif d'un côté, très négatif de l'autre.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(min_df=10, stop_words=list(STOP_FR))
X = vec.fit_transform(train["text"])
mots = np.array(vec.get_feature_names_out())

pos = np.asarray(X[(train["polarite"] == "positif").values].sum(axis=0)).ravel()
neg = np.asarray(X[(train["polarite"] == "négatif").values].sum(axis=0)).ravel()

# log-odds lissé : positif si > 0, négatif si < 0
logodds = np.log((pos + 1) / (pos.sum() + 1)) - np.log((neg + 1) / (neg.sum() + 1))
ordre = logodds.argsort()

print("Mots plutôt POSITIFS :", ", ".join(mots[ordre[::-1][:15]]))
print("Mots plutôt NÉGATIFS :", ", ".join(mots[ordre[:15]]))

Et visuellement, avec un **nuage de mots** par polarité :

In [ ]:
from wordcloud import WordCloud

def nuage(polarite, ax):
    texte = " ".join(train[train["polarite"] == polarite]["text"])
    wc = WordCloud(width=500, height=350, background_color="white",
                   stopwords=STOP_FR, colormap="viridis").generate(texte)
    ax.imshow(wc); ax.axis("off"); ax.set_title(f"Critiques {polarite}s")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
nuage("positif", axes[0])
nuage("négatif", axes[1])
plt.show()

### Hack Time

- Le nuage de mots est dominé par des mots fréquents mais peu informatifs ? Enrichis `STOP_FR` et relance.
- Affiche les 30 mots les plus négatifs au lieu de 15.

In [ ]:
# Votre code ici

## 4.2 Application 2 — Analyse de sentiment (modèle pré-entraîné)

*On monte d'un cran : on utilise un modèle déjà entraîné, en boîte noire.* La librairie `transformers` permet de charger en une ligne un modèle multilingue qui prédit un sentiment **sans qu'on l'ait entraîné**. On l'applique à un échantillon du test et on compare ses prédictions aux vraies étiquettes.

In [ ]:
from transformers import pipeline

analyseur = pipeline(
    "sentiment-analysis",
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
)

echantillon = test.sample(200, random_state=42).copy()
res = analyseur(echantillon["text"].tolist(), truncation=True, batch_size=16)

trad = {"positive": "positif", "negative": "négatif", "neutral": "neutre"}
echantillon["sentiment_predit"] = [trad[r["label"]] for r in res]

pd.crosstab(echantillon["polarite"], echantillon["sentiment_predit"],
            rownames=["vrai"], colnames=["prédit"])

Le modèle n'a **jamais vu** notre corpus et s'en sort déjà bien. Avantage : aucun entraînement. Limite : on ne contrôle ni ce qu'il a appris, ni ses catégories (il ajoute un `neutre` qui n'existe pas chez nous).

## 4.3 Application 3 — Classification supervisée (machine learning)

*On entraîne nous-mêmes un modèle.* On vectorise les critiques en **TF-IDF** (vu au notebook 1), puis on entraîne une **régression logistique** à prédire la polarité — exactement le cadre du machine learning supervisé vu en J2, mais avec du texte en entrée.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

tfidf = TfidfVectorizer(min_df=5, stop_words=list(STOP_FR))
X_train = tfidf.fit_transform(train["text"])
X_test = tfidf.transform(test["text"])

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train["polarite"])

pred = clf.predict(X_test)
print("Accuracy sur le test :", round(accuracy_score(test["polarite"], pred), 3))

ConfusionMatrixDisplay(
    confusion_matrix(test["polarite"], pred, labels=clf.classes_),
    display_labels=clf.classes_,
).plot(cmap="Blues"); plt.show()

Comme c'est un modèle **linéaire**, on peut lire ses coefficients : les mots qui poussent le plus vers « positif » ou « négatif ». On retrouve l'analyse de l'application 1, mais cette fois **apprise** pour maximiser la prédiction.

In [ ]:
vocab = np.array(tfidf.get_feature_names_out())
coef = clf.coef_[0]   # classes_ = ['négatif', 'positif'] -> coef pour 'positif'
ordre = coef.argsort()

print("Vers POSITIF :", ", ".join(vocab[ordre[::-1][:15]]))
print("Vers NÉGATIF :", ", ".join(vocab[ordre[:15]]))

### Hack Time

- Écris ta propre critique et fais-la prédire : `clf.predict(tfidf.transform(["..."]))`.
- Essaie une phrase ironique (« génial, j'ai adoré m'ennuyer ») : le modèle se laisse-t-il piéger ?

In [ ]:
# Votre code ici

## 4.4 Application 4 — Recherche sémantique

*On passe aux embeddings denses.* Plutôt que de chercher des mots exacts, on cherche par le **sens**. On encode chaque critique avec un modèle multilingue de phrases, puis on classe les critiques par **similarité cosinus** avec une requête en langage naturel. Une critique qui parle d'un « mauvais jeu d'acteur » ressortira même si elle n'emploie pas ces mots exacts.

In [ ]:
from sentence_transformers import SentenceTransformer, util

modele = SentenceTransformer("distiluse-base-multilingual-cased-v2")
corpus = test["text"].tolist()
emb_corpus = modele.encode(corpus, show_progress_bar=False, batch_size=64)

def rechercher(requete, k=3):
    emb = modele.encode([requete])
    score = util.cos_sim(emb, emb_corpus).numpy().ravel()
    for i in score.argsort()[::-1][:k]:
        print(f"[{score[i]:.2f}] ({test['polarite'].iloc[i]}) {corpus[i][:200]}...\n")

rechercher("un jeu d'acteur décevant et un scénario plat")

## 4.5 Application 5 — Biais éthiques

*La plus avancée : un regard critique sur les outils précédents.* Un modèle apprend les **régularités de ses données**, y compris leurs **biais**. Testons l'analyseur de sentiment sur des phrases **identiques** ne différant que par un mot (genre, origine) : un modèle neutre devrait donner le même score.

In [ ]:
paires = [
    "Le film est porté par un acteur talentueux.",
    "Le film est porté par une actrice talentueuse.",
    "Un personnage français au grand cœur.",
    "Un personnage algérien au grand cœur.",
    "Un personnage africain au grand cœur.",
]
for t, r in zip(paires, analyseur(paires)):
    print(f"{r['label']:>9} ({r['score']:.2f})  |  {t}")

Si les scores varient alors que **seul un mot d'identité change**, le modèle reflète des associations apprises dans ses données. Deux sources de biais à toujours garder en tête :

- **biais du modèle** : il a été pré-entraîné sur d'énormes corpus web, avec leurs stéréotypes ;
- **biais du corpus** : nos critiques Allociné sur-représentent certains publics, certains genres de films, une époque. Un modèle entraîné dessus **hérite** de ces déséquilibres.

Pour la recherche en SHS, ces outils sont précieux mais ne sont **pas neutres** : il faut documenter le corpus, tester la robustesse, et interpréter les résultats avec un regard critique.

## Récap module 4

| # | Application | Outil principal | Supervisé ? | Difficulté |
|---|---|---|---|---|
| 1 | Nuage de mots / mots-clés | `CountVectorizer`, `wordcloud` | non | ★ |
| 2 | Analyse de sentiment | `transformers` (pré-entraîné) | non | ★★ |
| 3 | Classification | TF-IDF + `LogisticRegression` | **oui** | ★★★ |
| 4 | Recherche sémantique | `sentence-transformers` | non | ★★★ |
| 5 | Biais éthiques | tout ce qui précède + esprit critique | — | ★★★★ |

Du simple comptage de mots au regard critique sur les biais, le text mining offre une **boîte à outils** pour explorer un corpus textuel.

→ Direction l'**après-midi** : on quitte le texte pour les données **audio et visuelles** (transcription, analyse d'images).